# **Conditional Chain**

In [1]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAI

In [2]:
llm_kimi = OpenAI(
    base_url='https://api.moonshot.ai/v1',
    api_key=os.environ.get('KIMI_API_KEY'),
    # model='kimi-k2.6'
    model='moonshot-v1-8k'
)

In [3]:
from pydantic import BaseModel
from typing import Literal
from langchain_core.output_parsers import PydanticOutputParser
class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]

# llm_structured_output = llm_kimi.with_structured_output(llm_schema)
llm_structured_output = PydanticOutputParser(pydantic_object=llm_schema)

In [4]:
initial_template =  ChatPromptTemplate.from_messages([
    ('system', 'You are a great movie reviewer'),
    ('user', 'Based on the input, categorize movies as positive or negative: {input}')
])
print('==initial_template')

==initial_template


In [5]:
from langchain_core.runnables import RunnableLambda
def pydantic_json(input: llm_schema):
    return input.model_dump()['movie_summary_flag']
pydantic_json_lambda = RunnableLambda(pydantic_json)

In [6]:
from langchain_core.output_parsers import StrOutputParser
str_parser = StrOutputParser()
def social_media_post_generator(text: str, platform):
    post_template = ChatPromptTemplate.from_messages([
        ('system', 'You are the best social media handler, You write catchy and lucrative posts for {platform}'),
        ('user', 'Write the best post possible for the {text} for the platform {platform}')
    ])

    chain = post_template | llm_kimi | str_parser
    return chain.invoke({'text': text, 'platform': platform})


In [7]:
from functools import partial
linkedin_chain = RunnableLambda(partial(social_media_post_generator, platform='Linkedin'))
insta_chain = RunnableLambda(partial(social_media_post_generator, platform='Instagram'))

In [8]:
from langchain_core.runnables import RunnableBranch
conditional_chain = RunnableBranch(
    (lambda x: 'positive' in x, linkedin_chain),
    insta_chain
)
final_orchestrator = (
    initial_template | 
    llm_kimi |
    llm_structured_output |
    pydantic_json_lambda | 
    conditional_chain
)

In [9]:
result = final_orchestrator.invoke({"input": "KGF was a great movie"})

PermissionDeniedError: Error code: 403 - {'error': {'message': 'The API you are accessing is not open', 'type': 'permission_denied_error'}}